### PyTorch CustomNet Exercises

Welcome to the PyTorch CustomNet exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import requests
from zipfile import ZipFile
from io import BytesIO

## Set Device

In [ ]:
# Check for access to GPU
torch.manual_seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using {} Device'.format(device))

Using cuda Device


## Download Dataset

In [ ]:
# Define the path to the dataset
dataset_path = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
# Send a GET request to the URL
response = requests.get(dataset_path)
# Check if the request was successful
if response.status_code == 200:
    # Open the downloaded bytes and extract them
    with ZipFile(BytesIO(response.content)) as zip_file:
        zip_file.extractall('/dataset')
    print('Download and extraction complete!')


Download and extraction complete!


## Transform and Load Dataset

In [ ]:
# Define transforms
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomCrop(size=64, padding=4, padding_mode='reflect'),
    transforms.RandomResizedCrop(size=64, scale=(0.08, 1.0), ratio=(0.75, 1.333), interpolation=2),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# Load TinyImageNet
train_dataset = ImageFolder('/dataset/tiny-imagenet-200/train', transform=transform)
test_dataset = ImageFolder('/dataset/tiny-imagenet-200/test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


In [ ]:
# Load the dataset
# tiny_imagenet_dataset_train = ImageFolder(root='/dataset/tiny-imagenet-200/train', transform=transform)
# tiny_imagenet_dataset_test = ImageFolder(root='/dataset/tiny-imagenet-200/test', transform=transform)

# transform = transforms.Compose([
#     transforms.Resize((224, 224)),  # Resize to fit the input dimensions of the network
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
# ])

## Define Custom Neural Network

In [ ]:
# # Define the custom neural network
# class CustomNet(nn.Module):
#     def __init__(self):
#         super(CustomNet, self).__init__()
#         # Define layers of the neural network
#         self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
#         self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
#         # Add more layers...
#         #self.fc1 = nn.Linear(..) # 200 is the number of classes in TinyImageNet

#     def forward(self, x):
#         # Define forward pass

#         return x


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class CustomNet(nn.Module):
    def __init__(self, num_classes=200):
        super(CustomNet, self).__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        # Max pooling layer
        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(256 * 8 * 8, 512)  # Assuming input image size is 64x64 after two max pooling
        self.fc2 = nn.Linear(512, num_classes)   # 200 is the number of classes in TinyImageNet

        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # Convolutional layers with activation and pooling
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        # Flatten the output for fully connected layers
        x = x.view(-1, 256 * 8 * 8)

        # Fully connected layers with dropout
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x


## Initialize the model

In [ ]:

# Initialize the model
model = CustomNet().to(device)
print(model)

CustomNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=16384, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=200, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [ ]:
X = torch.rand(1, 3, 64, 64, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f'pred class : {y_pred}')

pred class : tensor([33], device='cuda:0')


## Loss and optimizer

In [ ]:

# Loss and optimizer
#criterion = ...
#optimizer = ...
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

## Training Loop

In [ ]:
# Training loop
def train(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()

        # Step 1: Zero the gradients
        optimizer.zero_grad()

        # Step 2: Forward pass
        outputs = model(inputs)

        # Step 3: Compute loss
        loss = criterion(outputs, targets)

        # Step 4: Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

# Test loop
def test(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.cuda(), targets.cuda()

            # Forward pass
            outputs = model(inputs)

            # Compute loss
            loss = criterion(outputs, targets)

            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

    test_loss = test_loss / len(test_loader)
    test_accuracy = 100. * correct / total
    print(f'Test Loss: {test_loss:.6f} Acc: {test_accuracy:.2f}%')
    return test_accuracy

# Run the training and testing
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, criterion, optimizer)
test_accuracy = test(model, test_loader, criterion)
print(f'Final Test Accuracy: {test_accuracy:.2f}%')


Train Epoch: 1 Loss: 5.030593 Acc: 2.30%
Train Epoch: 2 Loss: 4.721860 Acc: 4.95%
Train Epoch: 3 Loss: 4.595908 Acc: 6.37%
Train Epoch: 4 Loss: 4.510663 Acc: 7.31%
Train Epoch: 5 Loss: 4.459516 Acc: 7.95%
Train Epoch: 6 Loss: 4.419124 Acc: 8.47%
Train Epoch: 7 Loss: 4.391170 Acc: 8.83%
Train Epoch: 8 Loss: 4.365654 Acc: 9.09%
Train Epoch: 9 Loss: 4.338107 Acc: 9.57%
Train Epoch: 10 Loss: 4.325162 Acc: 9.73%
Test Loss: 7.333964 Acc: 1.15%
Final Test Accuracy: 1.15%


In [ ]:
#model = models.vgg16(pretrained=true)
model_path = 'custom_net_model1.pth'

# Saving the model after training
torch.save(model.state_dict(), model_path)
print(f"Model saved to {model_path}")

Model saved to custom_net_model1.pth


In [ ]:




# # Training loop
# def train(epoch, model, train_loader, criterion, optimizer):
#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0

#     for batch_idx, (inputs, targets) in enumerate(train_loader):
#         inputs, targets = inputs.cuda(), targets.cuda()
#         # todo
#         '''
#         What's the correct order of those instructions?
#         1. optimizer.zero_grad()
#         2. loss = criterion(outputs, targets)
#         3. outputs = model(inputs)
#         4. optimizer.step()
#         5. loss.backward()

#         '''
#         running_loss += loss.item()
#         _, predicted = outputs.max(1)
#         total += targets.size(0)
#         correct += predicted.eq(targets).sum().item()

#     train_loss = running_loss / len(train_loader)
#     train_accuracy = 100. * correct / total
#     print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

# # Test loop
# def test(model, test_loader, criterion):
#     model.eval()
#     test_loss = 0
#     correct = 0
#     total = 0
#     with torch.no_grad():
#         for batch_idx, (inputs, targets) in enumerate(test_loader):
#             inputs, targets = inputs.cuda(), targets.cuda()
#             outputs = model(inputs)
#             loss = criterion(outputs, targets)

#             test_loss += loss.item()
#             _, predicted = outputs.max(1)
#             total += targets.size(0)
#             correct += predicted.eq(targets).sum().item()

#     test_loss = test_loss / len(test_loader)
#     test_accuracy = 100. * correct / total
#     print(f'Test Loss: {test_loss:.6f} Acc: {test_accuracy:.2f}%')
#     return test_accuracy

# # Run the training and testing
# num_epochs = 10
# for epoch in range(1, num_epochs + 1):
#     train(epoch, model, train_loader, criterion, optimizer)
# test_accuracy = test(model, test_loader, criterion)